# 03 Experimentation

            This notebook compares Random Forest, Gradient Boosting, Logistic Regression, and LDA under two scenarios: one production-safe scenario without `duration`, and one diagnostic scenario with `duration`.

            The evaluation uses accuracy, sensitivity, specificity, and balanced accuracy. Sensitivity is calculated for the `no` class and specificity for the `yes` class to match the confusion-matrix convention used in the training code.


## Set Paths for the Experiment

            The model-comparison workflow writes both a YAML artifact and a CSV table. The CSV is convenient for quick notebook inspection, while the YAML keeps additional detail for tracking and reproducibility.


In [11]:
# Define paths for raw data and model-comparison outputs.

from pathlib import Path
import subprocess
import sys
import pandas as pd
import logging

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'bank.csv'
COMPARISON = PROJECT_ROOT / 'models' / 'trained' / 'bank_model_comparison.yaml'


## Model Comparison

            To keep the notebook run time manageable, this command uses 20 Monte Carlo iterations and the selected GBM configuration. For a more exhaustive experiment, the iteration count can be increased and `--full-gbm-grid` can be added.


In [12]:
# Run Random Forest, GBM, Logistic Regression, and LDA for both duration scenarios.

logging.disable(logging.INFO)   # suppress INFO logs
subprocess.run([
    sys.executable, str(PROJECT_ROOT / 'src' / 'models' / 'compare_models.py'),
    '--raw-data', str(RAW_DATA),
    '--output', str(COMPARISON),
    '--mccv-iterations', '20',
    '--rf-trees', '500',
], check=True)
logging.disable(logging.NOTSET)  # re-enable logging


The comparison workflow evaluates both scenarios and saves the results under `models/trained/`. Keeping repeated model-training logic in project code makes the notebook easier to read and easier to reproduce.


## Compare Model Results

            The results are sorted by scenario and balanced accuracy so the strongest model in each setting is easy to identify.


In [13]:
# Load and sort the model-comparison table.

comparison_table = pd.read_csv(COMPARISON.with_suffix('.csv'))
comparison_table.sort_values(['scenario', 'balanced_accuracy'], ascending=[True, False])


,scenario,model,accuracy,sensitivity,specificity,balanced_accuracy,best_mtry,n_trees,best_learning_rate,best_max_depth,best_min_samples_leaf,best_n_estimators
1,No Duration,GBM,0.722794,0.795745,0.641777,0.718761,NaN,NaN,0.05,7.0,5.0,500.0
0,No Duration,Random Forest,0.713390,0.777872,0.641777,0.709825,3.0,500.0,NaN,NaN,NaN,NaN
2,No Duration,Logistic Regression,0.696373,0.765106,0.620038,0.692572,NaN,NaN,NaN,NaN,NaN,NaN
3,No Duration,LDA,0.696820,0.816170,0.564272,0.690221,NaN,NaN,NaN,NaN,NaN,NaN
5,With Duration,GBM,0.852217,0.827234,0.879962,0.853598,NaN,NaN,0.05,7.0,5.0,500.0
4,With Duration,Random Forest,0.844156,0.812766,0.879017,0.845891,6.0,500.0,NaN,NaN,NaN,NaN
6,With Duration,Logistic Regression,0.819525,0.841702,0.794896,0.818299,NaN,NaN,NaN,NaN,NaN,NaN
7,With Duration,LDA,0.806986,0.856170,0.752363,0.804267,NaN,NaN,NaN,NaN,NaN,NaN


Without `duration`, GBM performs best with about 0.723 accuracy and 0.719 balanced accuracy, while Random Forest is close behind. Logistic Regression and LDA are lower, which suggests the ensemble models capture nonlinear structure better.

            When `duration` is included, all models improve substantially. GBM rises to about 0.852 accuracy and 0.854 balanced accuracy. This confirms that call duration is highly predictive, but it remains excluded from production scoring because it is not available before a call ends.


## Train the Production Model

            After comparing models, the final production model is trained on the no-duration feature matrix. This model is saved as `bank_deposit_model.pkl` and served by FastAPI and Streamlit.


In [14]:
# Train and save the production no-duration Gradient Boosting model.

logging.disable(logging.INFO)   # suppress INFO logs
subprocess.run([
    sys.executable, str(PROJECT_ROOT / 'src' / 'models' / 'train_model.py'),
    '--config', str(PROJECT_ROOT / 'configs' / 'model_config.yaml'),
    '--data', str(PROJECT_ROOT / 'data' / 'processed' / 'featured_bank_data.csv'),
    '--models-dir', str(PROJECT_ROOT / 'models'),
], check=True)
logging.disable(logging.NOTSET)  # re-enable logging


The training script also writes a metrics file so model performance can be checked without reopening the notebook.


## Review Final Production Metrics

            The saved metrics artifact documents the final model result used by the application layer.


In [15]:
# Load the saved production metrics artifact.

import yaml

metrics_path = PROJECT_ROOT / 'models' / 'trained' / 'bank_deposit_model_metrics.yaml'
yaml.safe_load(metrics_path.read_text())


{'model': 'bank_deposit_model',
 'algorithm': 'GradientBoosting',
 'target_variable': 'deposit',
 'metrics': {'accuracy': 0.7227944469323779,
  'sensitivity': 0.7957446808510639,
  'specificity': 0.6417769376181475,
  'balanced_accuracy': 0.7187608092346056,
  'precision_yes': 0.7388465723612623,
  'recall_yes': 0.6417769376181475,
  'f1': 0.6868993424380375,
  'roc_auc': 0.7846583276354422},
 'dependencies': {'python_version': '3.11.9',
  'scikit_learn_version': '1.2.2',
  'pandas_version': '1.5.3',
  'numpy_version': '1.24.3'}}

The final production GBM gives about 0.723 accuracy, 0.719 balanced accuracy, 0.687 F1, and 0.785 ROC AUC. This is the no-duration result, which better represents a realistic customer-targeting model rather than a post-call explanatory model.
